# Write a calculation

[A first loop](../quickstart.ipynb) solves a system built from the library's components. A `Calculation` is the smallest unit of such a system: a class that owns a few rows of the state vector $y$ and returns those rows of $M\,\dot y = F(y, t)$. This page writes one, a lumped block heated at a given rate and cooled to a fixed environment, solves its steady state and runs a short transient, each checked by hand.

The block has two rows: its temperature `T`, which integrates the heat balance, and the heat `Q_out` it loses through a conductance, which balances at every instant. Its heating `Q_in` comes from outside.

In [1]:
import numpy as np

from stream import Aggregator, Calculation, State, unpacked
from stream.analysis.report import report

## The class

Subclass `Calculation` and give it four members. `name` is the calculation's key in every `State`, and two calculations of one name make the system refuse to build. `variables` maps each row's name to its place, 0-based within this calculation's own rows. `mass_vector` has one entry per row: `True` for a differential row ($\dot y_i = F_i$), `False` for an algebraic one ($F_i = 0$). `calculate` receives its own rows first and every other input as a keyword, and returns its rows of $F$.

Each keyword arrives as a small dictionary, keyed by the supplying calculation for an input from another calculation and by the receiving one for a `funcs` input; `@unpacked` turns it into a plain number or array. The trailing `**_` absorbs inputs the method does not ask for.

In [2]:
class HeatedBlock(Calculation):
    def __init__(self, name, capacity, conductance, T_env):
        self.name = name
        self.capacity = capacity
        self.conductance = conductance
        self.T_env = T_env

    @property
    def variables(self):
        return {"T": 0, "Q_out": 1}

    @property
    def mass_vector(self):
        return (True, False)

    @unpacked
    def calculate(self, variables, *, Q_in, **_):
        T, Q_out = variables
        return np.array(
            [
                (Q_in - Q_out) / self.capacity,
                self.conductance * (T - self.T_env) - Q_out,
            ]
        )

Two contracts hold. The return has exactly as many rows as the calculation owns, in `variables` order: row 0 is the equation of `T` because `T` sits at place 0, and a correct expression in the wrong row solves a different system without complaint. The first argument is a view into the solver's own vector: it is read, never written to, and the return is a new array.

## Build it alone and read the report

`Aggregator.from_decoupled` makes a system of calculations with no links between them, and `funcs` hands a calculation its inputs from outside: a constant, or a callable of time. A `funcs` input costs no rows. `report` lists each calculation's rows and inputs.

In [3]:
block = HeatedBlock("block", capacity=18e3, conductance=25.0, T_env=40.0)
agr = Aggregator.from_decoupled(block, funcs={block: {"Q_in": 1000.0}})
report(agr)

Contains 2 Equations, 1 Nodes, 0 Edges.

| Calculation | Type | Equation No. | Unset | Set Externally | Missing |
| :---------- | :--- | :----------- | :---- | :------------- | :------ |
| block | HeatedBlock | 0 - 2 |  | Q_in |  |


`Q_in` is **Set Externally**, and the **Missing** column is empty. Read that column before every solve: an input listed there has no supplier and no default, and the solve dies with a `TypeError` at the first evaluation of $F$. **Unset** lists every input without a supplier, so a keyword with a default appears there alone and runs on its default, and a **Missing** input appears in both. `funcs` is keyed by the calculation object, not by its name: a dictionary keyed by the string `"block"` reaches nothing, and `Q_in` turns up **Missing**.

## Solve the steady state

`solve_steady` takes a guess as a `State`, keyed by calculation name and then row name, and returns the solved vector. `agr.save` turns that vector back into a `State`, and `agr.compute` evaluates $F$ there. At steady state the block loses what it receives, so $T = T_{env} + Q_{in} / G$, with $G$ the conductance.

In [4]:
y = agr.solve_steady(State({"block": {"T": 40.0, "Q_out": 0.0}}))
steady = agr.save(y)["block"]
print(f"T solved:     {steady['T']:.4f} °C")
print(f"T by hand:    {block.T_env + 1000.0 / block.conductance:.4f} °C")
print(f"Q_out solved: {steady['Q_out']:.4f} W")
print(f"residual:     {agr.compute(y)}")

T solved:     80.0000 °C
T by hand:    80.0000 °C
Q_out solved: 1000.0000 W
residual:     [0. 0.]


The block settles on the hand answer, 80 °C to every printed digit, with zero residual in both rows.

## Run a transient

The class takes a step in its heating unchanged: the input becomes a callable, here 1000 W for the first 10 s and 1500 W after. `solve` takes the steady vector and a time grid; one differential row and one algebraic row make it a DAE solve. After the step the temperature approaches its new steady state with the time constant $\tau = C / G$ = 720 s, so by hand $T(t) = 80 + (500 / G)\,(1 - e^{-(t - 10)/\tau})$.

In [5]:
agr_step = Aggregator.from_decoupled(
    block, funcs={block: {"Q_in": lambda t: 1000.0 if t < 10.0 else 1500.0}}
)
time = np.linspace(0.0, 60.0, 61)
sol = agr_step.solve(y, time)
T_end = agr_step.save(sol.data[-1])["block"]["T"]
tau = block.capacity / block.conductance
print(f"T(60 s) solved:  {T_end:.4f} °C")
print(f"T(60 s) by hand: {80.0 + 500.0 / block.conductance * (1.0 - np.exp(-50.0 / tau)):.4f} °C")
print(f"share of the way to 100 °C: {(T_end - 80.0) / 20.0:.3f}")

T(60 s) solved:  81.3418 °C
T(60 s) by hand: 81.3418 °C
share of the way to 100 °C: 0.067


At 60 s the block is at about 81.34 °C, where the exponential puts it: 50 s after the step it has covered about 7 % of the way to its new steady state of 100 °C.

[Wire a flow network](wire-a-flow-network.ipynb) is the next task: the library's own calculations, joined by a flow graph instead of fed by `funcs`. The [Coupled Masses and Springs](../CoupledSprings.ipynb) example writes a calculation of three rows and links two of them by graph edges, each reading the other's rows as keywords.